<a href="https://colab.research.google.com/github/harjussingh/TIRP-Team02/blob/main/Symptom_Severness_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import BernoulliNB
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, VotingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import warnings
warnings.filterwarnings("ignore")

In [2]:
INPUT_CSV = "/content/drive/MyDrive/Dataset/disease_dataset.csv"
df = pd.read_csv(INPUT_CSV)

In [3]:
print(f"Dataset shape  : {df.shape}")
print(f"Total diseases : {df['disease'].nunique()}")
print(f"ATS levels     : {sorted(df['ats_level'].unique())}")

Dataset shape  : (15000, 238)
Total diseases : 75
ATS levels     : [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]


# **Pre-processing**

In [4]:
META_COLS = ["disease", "ats_level", "ats_label", "advice"]

**Separate features and labels**

In [5]:
symptom_cols = [c for c in df.columns if c not in META_COLS]

X = df[symptom_cols].astype(int)
y_disease = df["disease"]
y_ats     = df["ats_level"]

In [6]:
le_disease = LabelEncoder()
y_encoded  = le_disease.fit_transform(y_disease)

In [7]:
ats_lookup = df.drop_duplicates("disease").set_index("disease")[["ats_level","ats_label","advice"]]

print(f"Feature matrix : {X.shape}")
print(f"Classes        : {len(le_disease.classes_)}")

Feature matrix : (15000, 234)
Classes        : 75


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=0.2,
    stratify=y_encoded,
    random_state=42
)

print(f"X_train : {X_train.shape}   y_train : {y_train.shape}")
print(f"X_test  : {X_test.shape}    y_test  : {y_test.shape}")

X_train : (12000, 234)   y_train : (12000,)
X_test  : (3000, 234)    y_test  : (3000,)


In [9]:
# 5a. Naive Bayes (Bernoulli — ideal for binary features)
nb_model = BernoulliNB(alpha=0.5)
nb_model.fit(X_train, y_train)
print(f"Naive Bayes   accuracy: {accuracy_score(y_test, nb_model.predict(X_test)):.4f}")

# 5b. Decision Tree
dt_model = DecisionTreeClassifier(max_depth=20, min_samples_split=5, random_state=42)
dt_model.fit(X_train, y_train)
print(f"Decision Tree accuracy: {accuracy_score(y_test, dt_model.predict(X_test)):.4f}")

# 5c. Random Forest
rf_model = RandomForestClassifier(n_estimators=200, max_depth=None,
                                   min_samples_split=4, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
print(f"Random Forest accuracy: {accuracy_score(y_test, rf_model.predict(X_test)):.4f}")

# 5d. Extra Trees
et_model = ExtraTreesClassifier(n_estimators=200, min_samples_split=4,
                                 random_state=42, n_jobs=-1)
et_model.fit(X_train, y_train)
print(f"Extra Trees   accuracy: {accuracy_score(y_test, et_model.predict(X_test)):.4f}")


Naive Bayes   accuracy: 0.9940
Decision Tree accuracy: 0.3973
Random Forest accuracy: 0.9930
Extra Trees   accuracy: 0.9920


In [10]:
# ── 6. WEIGHTED ENSEMBLE ─────────────────────────────────────────────────────
"""
Weights match your existing code: rf=1, et=2, nb=3, dt=1  (total=7)
Using predict_proba and averaging — same logic you already have.
"""

def ensemble_predict(X_input):
    probs = (
        1 * rf_model.predict_proba(X_input) +
        2 * et_model.predict_proba(X_input) +
        3 * nb_model.predict_proba(X_input) +
        1 * dt_model.predict_proba(X_input)
    ) / 7
    return probs

ensemble_preds = np.argmax(ensemble_predict(X_test), axis=1)
print(f"Ensemble      accuracy: {accuracy_score(y_test, ensemble_preds):.4f}")



Ensemble      accuracy: 0.9947


In [11]:
print("\n── Classification Report (Ensemble) ──")
print(classification_report(
    y_test, ensemble_preds,
    target_names=le_disease.classes_,
    zero_division=0
))


── Classification Report (Ensemble) ──
                                precision    recall  f1-score   support

  Acute Angle-Closure Glaucoma       1.00      1.00      1.00        40
           Acute Heart Failure       1.00      1.00      1.00        40
          Acute Kidney Failure       1.00      1.00      1.00        40
          Acute Limb Ischaemia       1.00      1.00      1.00        40
   Acute Myocardial Infarction       0.93      1.00      0.96        40
          Acute Pyelonephritis       1.00      1.00      1.00        40
     Acute Respiratory Failure       1.00      1.00      1.00        40
               Acute Sinusitis       1.00      1.00      1.00        40
       Acute Urinary Retention       1.00      1.00      1.00        40
                 Acute Vertigo       1.00      1.00      1.00        40
            Anaphylactic Shock       1.00      1.00      1.00        40
                  Ankle Sprain       1.00      1.00      1.00        40
             Aortic Dis

In [12]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(rf_model, X, y_encoded, cv=cv, scoring="accuracy", n_jobs=-1)
print(f"Random Forest 5-fold CV: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

Random Forest 5-fold CV: 0.9913 ± 0.0011


In [13]:
def predict_disease(symptom_list: list[str]) -> dict:
    """
    symptom_list : list of symptom strings the patient reports
                   e.g. ["fever", "cough", "chest pain"]

    Returns a dict with top-3 predictions + ATS advice.
    """
    vec = pd.DataFrame([[0] * len(symptom_cols)], columns=symptom_cols)
    for s in symptom_list:
        s = s.lower().strip()
        if s in vec.columns:
            vec[s] = 1

    probs  = ensemble_predict(vec)[0]
    top3_i = np.argsort(probs)[::-1][:3]

    results = []
    for i in top3_i:
        disease = le_disease.classes_[i]
        info    = ats_lookup.loc[disease] if disease in ats_lookup.index else None
        results.append({
            "disease"    : disease,
            "probability": round(float(probs[i]), 4),
            "ats_level"  : int(info["ats_level"])  if info is not None else None,
            "ats_label"  : info["ats_label"]        if info is not None else None,
            "advice"     : info["advice"]            if info is not None else None,
        })
    return results


# Example
sample = ["fever", "cough", "chest pain", "difficulty breathing"]
preds  = predict_disease(sample)
print("\n── Example Inference ──")
for p in preds:
    print(f"  {p['disease']:<35} prob={p['probability']:.3f}  ATS={p['ats_level']}  → {p['advice']}")


# ── 10. SAVE MODELS (optional) ───────────────────────────────────────────────
import pickle, os

os.makedirs("saved_models", exist_ok=True)
for name, model in [("rf",rf_model),("et",et_model),("nb",nb_model),("dt",dt_model),("le",le_disease)]:
    with open(f"saved_models/{name}_model.pkl","wb") as f:
        pickle.dump(model, f)

print("\nModels saved to saved_models/")


── Example Inference ──
  Severe Pneumonia                    prob=0.381  ATS=3  → Seek urgent medical care
  Hypertensive Crisis                 prob=0.153  ATS=1  → Call emergency services immediately
  Pleuritis                           prob=0.151  ATS=4  → Visit a GP

Models saved to saved_models/


In [14]:
pip install tensorflow --quiet